### Live Replay

In [1]:
import databento as db
from dotenv import load_dotenv
import pandas as pd
import os
load_dotenv()

with open("../../../universe.txt", 'r') as f:
    UNIVERSE = [line.strip() for line in f]
START = pd.Timestamp(2026, 6, 4, 9, 30, tz="US/Eastern")
END = pd.Timestamp(2026, 6, 4, 15, 50, tz="US/Eastern")

class Session_Volume_Accumulator:
    def __init__(self):
        self.counts = {}

    def counter(self, record: db.DBNRecord) -> None:
        if isinstance(record, db.MBP1Msg):
            id = record.instrument_id
            size = record.size
            if id not in self.counts.keys():
                self.counts[id] = size
            else:
                self.counts[id] += size
    
    def printer(self, record: db.DBNRecord) -> None:
        if isinstance(record, db.MBP1Msg):
            if record.size >= 500 and record.instrument_id == 33:
                print(record.size)
    
    def error_handler(self, exception: Exception) -> None:
        print(f"Exception: {exception}")

# Create a live client
client = db.Live(key=os.getenv("DATABENTO_API_KEY"))

# Subscribe with a specified start time for intraday replay
client.subscribe(
    dataset="EQUS.MINI",
    schema="tbbo",
    symbols=UNIVERSE,
    start=START,
)

acc = Session_Volume_Accumulator()
client.add_callback(record_callback=acc.counter,
                    exception_callback=acc.error_handler)
client.add_callback(record_callback=acc.printer)
client.start()
client.block_for_close(15)
client.stop()

600
585
1000
1000
570
677
700
566
1000
559
525
1000
700
1000
804
1000
1000
1000
839
649
639
543
846
500
1000
1000
565
955
600
1200
500
500
500
500
500
600
500
500
1152
840


In [2]:
acc.counts

{13340: 989127,
 16131: 1883514,
 20433: 274069,
 19729: 88634,
 11667: 6046797,
 10865: 2956031,
 13233: 581437,
 7151: 902904,
 10394: 439388,
 6297: 149302,
 17684: 1261278,
 16875: 68018,
 16244: 701633,
 1429: 2689323,
 262: 99982,
 33: 471176,
 16459: 88962,
 2994: 514385,
 5314: 299426,
 10924: 603409,
 17692: 1369319,
 773: 864486,
 18114: 18309,
 17200: 145436,
 2896: 151417,
 21095: 20672,
 58: 540145,
 13263: 402596,
 14261: 351405,
 10852: 247900,
 7756: 104212,
 14820: 479568,
 3356: 6088159,
 981: 166388,
 20790: 21156,
 2351: 120471,
 10345: 479783,
 13345: 112183,
 11872: 161390,
 20270: 35529,
 6823: 92332,
 3728: 277904,
 7152: 1559186,
 10888: 878315,
 1344: 25177,
 6905: 381868,
 16248: 1324900,
 10451: 783119,
 853: 1914759,
 5683: 12518,
 10724: 321421,
 8147: 4160156,
 8705: 1224719,
 17448: 1052062,
 38: 1311525,
 15002: 445036,
 2021: 393617,
 9976: 57402,
 8676: 125503,
 7716: 1071715,
 5960: 180102,
 3667: 326067,
 13760: 334716,
 10996: 1132899,
 3548: 35528

### Looking at data costs

In [ ]:
import databento as db
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
import os
START = pd.Timestamp(2022,6,10,00,00,00)
END = pd.Timestamp(2024,6,10,00,00,00)
DATASET = "XNAS.ITCH"
# SCHEMAS = ["imbalance", "bbo-1m", "ohlcv-1m", "statistics"]
SCHEMAS = ["ohlcv-1d"]
UNIVERSE = "ALL_SYMBOLS"
# with open("../universe_russell.txt", "r") as f:
#     UNIVERSE = [line.strip() for line in f]


client = db.Historical(os.getenv("DATABENTO-API-KEY"))
total_cost = 0
total_size = 0

for schema in SCHEMAS:
    cost = client.metadata.get_cost(
        dataset=DATASET,
        symbols=UNIVERSE,
        start=START,
        end=END,
        schema=schema
    )

    size = client.metadata.get_billable_size(
        dataset=DATASET,
        symbols=UNIVERSE,
        start=START,
        end=END,
        schema=schema
    )
    print(f"{schema} Cost: ${cost:,.2f}, Uncompressed Size: {size / 1e9:,.2f} GB")
    total_cost += cost
    total_size += size

print(f"Total Cost: ${total_cost:,.2f}, Uncompressed Size: {total_size / 1e9:,.2f} GB")

ohlcv-1d Cost: $0.00, Uncompressed Size: 0.27 GB
Total Cost: $0.00, Uncompressed Size: 0.27 GB
